# CodeQL — semantic static analysis (raw CLI output)

This notebook uses **`gh codeql`** (GitHub CLI + **gh-codeql** extension), the same as your Windows shell, e.g. toolchain **2.25.1** under:

`%LocalAppData%\GitHub CLI\extensions\gh-codeql\dist\release\…`

**Jupyter tip:** the extension’s wrapper script calls `gh` again. If the notebook kernel’s `PATH` does not include the GitHub CLI folder (`C:\Program Files\GitHub CLI` by default), you get `gh: command not found` inside the wrapper. The first code cell fixes that by prepending `gh.exe`’s directory to `PATH` for every subprocess.

Flow:

1. `gh codeql version` — same as your CLI.
2. `gh codeql resolve languages` — confirm **python** is visible.
3. Tiny sample project → **`gh codeql database create my-db --language=python --source-root .`** (`cwd` = project dir).
4. **`gh codeql pack download codeql/python-queries`** so suites resolve (matches toolchain help text).
5. **`gh codeql database analyze`** → SARIF, then JSON peek + `interpret-results` CSV.

References: [gh codeql](https://cli.github.com/manual/gh_codeql), [database create](https://docs.github.com/en/code-security/codeql-cli/codeql-cli-manual/database-create).

In [3]:
# GitHub CLI + gh codeql (Windows-friendly: PATH fix for the extension wrapper)
import os
import shutil
import subprocess


def find_gh_exe():
    w = shutil.which('gh')
    if w:
        return w
    candidates = [
        os.path.join(os.environ.get('ProgramFiles', r'C:\Program Files'), 'GitHub CLI', 'gh.exe'),
        os.path.join(os.environ.get('ProgramFiles(x86)', r'C:\Program Files (x86)'), 'GitHub CLI', 'gh.exe'),
    ]
    for c in candidates:
        if c and os.path.isfile(c):
            return c
    return None


def env_with_gh_on_path(base_env=None):
    """Prepend gh.exe’s directory so gh-codeql’s inner shell can find `gh`."""
    env = (base_env or os.environ).copy()
    gh_dir = os.path.dirname(os.path.abspath(GH))
    sep = os.pathsep
    current = env.get('PATH', '')
    parts = current.split(sep) if current else []
    if gh_dir not in parts:
        env['PATH'] = gh_dir + sep + current
    return env


def gh_codeql_run(argv_tail, *, cwd=None, timeout=None):
    """Run `gh codeql ...` with PATH fixed for the extension wrapper."""
    return subprocess.run(
        [GH, 'codeql', *argv_tail],
        cwd=cwd,
        env=CODEQL_ENV,
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace',
        timeout=timeout,
    )


GH = find_gh_exe()
if not GH:
    raise SystemExit(
        'gh.exe not found. Install GitHub CLI and/or add it to PATH. See https://cli.github.com/'
    )

CODEQL_ENV = env_with_gh_on_path()

r = gh_codeql_run(['version'])
print('gh:', GH)
print(r.stdout.strip() or r.stderr.strip())
if r.returncode != 0:
    raise SystemExit(f"gh codeql version failed (exit {r.returncode}). STDERR:\n{r.stderr}")

r2 = gh_codeql_run(['resolve', 'languages'])
print('\n--- gh codeql resolve languages ---')
print((r2.stdout or r2.stderr).strip())
if r2.returncode != 0:
    raise SystemExit(f"resolve languages failed (exit {r2.returncode})")
_langs = (r2.stdout or '').lower()
assert 'python' in _langs, 'Python extractor not visible; fix CodeQL install / extension.'
print('\nOK: python is listed for analysis.')

gh: C:\Program Files\GitHub CLI\gh.exe
CodeQL command-line toolchain release 2.25.1.
Copyright (C) 2019-2026 GitHub, Inc.
Unpacked in: C:\Users\vinod\AppData\Local\GitHub CLI\extensions\gh-codeql\dist\release\v2.25.1
   Analysis results depend critically on separately distributed query and
   extractor modules. To list modules that are visible to the toolchain,
   use 'codeql resolve packs' and 'codeql resolve languages'.

--- gh codeql resolve languages ---
actions (C:\Users\vinod\AppData\Local\GitHub CLI\extensions\gh-codeql\dist\release\v2.25.1\actions)
cpp (C:\Users\vinod\AppData\Local\GitHub CLI\extensions\gh-codeql\dist\release\v2.25.1\cpp)
csharp (C:\Users\vinod\AppData\Local\GitHub CLI\extensions\gh-codeql\dist\release\v2.25.1\csharp)
csv (C:\Users\vinod\AppData\Local\GitHub CLI\extensions\gh-codeql\dist\release\v2.25.1\csv)
go (C:\Users\vinod\AppData\Local\GitHub CLI\extensions\gh-codeql\dist\release\v2.25.1\go)
html (C:\Users\vinod\AppData\Local\GitHub CLI\extensions\gh-codeq

In [4]:
# Minimal Python project for a reproducible database
import os
import tempfile
import textwrap

work = tempfile.mkdtemp(prefix='codeql_nb_')
src = os.path.join(work, 'sample.py')
with open(src, 'w', encoding='utf-8') as f:
    f.write(textwrap.dedent('''\
        """Tiny sample for CodeQL Python extraction."""
        import os

        def greet(name: str) -> str:
            return f"hello {name}"

        def unsafe_env(key: str):
            # Intentionally reads env without validation (may match some queries)
            return os.environ.get(key)
    '''))

db_name = 'my-db'
db_dir = os.path.join(work, db_name)
sarif_out = os.path.join(work, 'results.sarif')
csv_out = os.path.join(work, 'interpret.csv')
print('work:', work)
print('database dir (relative to work):', db_name)
print('database path:', db_dir)

work: C:\Users\vinod\AppData\Local\Temp\codeql_nb_1use4tkl
database dir (relative to work): my-db
database path: C:\Users\vinod\AppData\Local\Temp\codeql_nb_1use4tkl\my-db


In [5]:
# gh codeql database create my-db --language=python --source-root .
r = gh_codeql_run(
    ['database', 'create', db_name, '--language=python', '--source-root', '.', '--overwrite'],
    cwd=work,
    timeout=600,
)
print('return code:', r.returncode)
if r.stdout:
    print('STDOUT:\n', r.stdout)
if r.stderr:
    print('STDERR:\n', r.stderr)
assert r.returncode == 0, 'database create failed; see stderr above'

return code: 2
STDOUT:
 [2026-03-27 18:11:41] [build-stderr] Unable to create process using 'C:\Python\python.exe --version': The system cannot find the file specified.
[2026-03-27 18:11:41] [build-stderr] The `py` launcher is required for CodeQL to work on Windows.Please include it when installing Python for Windows.see https://docs.python.org/3/using/windows.html#python-launcher-for-windows
[2026-03-27 18:11:41] [ERROR] Spawned process exited abnormally (code 4; tried to run: [C:\Users\vinod\AppData\Local\GitHub CLI\extensions\gh-codeql/dist/release/v2.25.1\tools\win64\runner.exe, cmd.exe, /C, type, NUL, &&, C:\Users\vinod\AppData\Local\GitHub CLI\extensions\gh-codeql\dist\release\v2.25.1\python\tools\autobuild.cmd])

STDERR:
 Initializing database at C:\Users\vinod\AppData\Local\Temp\codeql_nb_1use4tkl\my-db.
Running build command: []
Running command in C:\Users\vinod\AppData\Local\Temp\codeql_nb_1use4tkl: [C:\Users\vinod\AppData\Local\GitHub CLI\extensions\gh-codeql\dist\release\v2

AssertionError: database create failed; see stderr above

In [ ]:
# Download Python query pack (toolchain message: use resolve packs / pack download)
r = gh_codeql_run(['pack', 'download', 'codeql/python-queries'], timeout=600)
print('return code:', r.returncode)
if r.stdout:
    print('STDOUT:\n', r.stdout.strip()[:4000])
if r.stderr:
    print('STDERR:\n', r.stderr.strip()[:4000])
assert r.returncode == 0, 'pack download failed; check network / gh auth if registry requires it'

In [ ]:
# Analyze → SARIF
suite = os.environ.get(
    'CODEQL_PYTHON_SUITE',
    'codeql/python-queries:codeql-suites/python-code-scanning.qls',
)

r = gh_codeql_run(
    [
        'database', 'analyze', db_name,
        '--format=sarif-latest',
        f'--output={sarif_out}',
        suite,
    ],
    cwd=work,
    timeout=600,
)
print('return code:', r.returncode)
print('suite:', suite)
if r.stdout:
    print('STDOUT:\n', r.stdout)
if r.stderr:
    print('STDERR:\n', r.stderr)
assert r.returncode == 0, 'database analyze failed'
print('SARIF path:', sarif_out)
print('SARIF bytes:', os.path.getsize(sarif_out))

In [ ]:
# Structural summary of raw SARIF JSON
import json

raw = open(sarif_out, encoding='utf-8').read()
doc = json.loads(raw)

print('top-level keys:', sorted(doc.keys()))
runs = doc.get('runs', [])
print('runs:', len(runs))
if runs:
    r0 = runs[0]
    print('run[0] keys:', sorted(r0.keys()))
    res = r0.get('results', [])
    print('results in run[0]:', len(res))
    if res:
        print('result[0] keys:', sorted(res[0].keys()))

print('\n--- raw SARIF (full JSON) ---')
print(json.dumps(doc, indent=2))

In [ ]:
# interpret-results → CSV
r = gh_codeql_run(
    [
        'database', 'interpret-results',
        '--format=csv',
        f'--output={csv_out}',
        '--', db_name, suite,
    ],
    cwd=work,
    timeout=600,
)
print('return code:', r.returncode)
if r.stdout:
    print('STDOUT:\n', r.stdout)
if r.stderr:
    print('STDERR:\n', r.stderr)
print('--- raw CSV (first 8000 chars) ---')
print(open(csv_out, encoding='utf-8').read()[:8000])